# **PARTE B – Regresión logística**

## **Práctica**
El riesgo de disconfort térmico o sobrecalentamiento en una sala de reuniones de la Facultad de
Informática depende de diversos factores, como por ejemplo la temperatura ambiente y la humedad
relativa.

---

1. Genera un dataset sintético que contenga 1000 instancias que representen mediciones de una sala inteligente en distintos momentos del tiempo. Cada instancia debe incluir los siguientes atributos:

    * **Temperatura ambiente**. Ten en cuenta que en esa sala, la temperatura suele encontrarse en condiciones normales: entre $20\ y\ 24\ °C$, en situaciones no óptimas: entre $12\ y\ 30\ °C$.
    * **Humedad relativa**. La humedad relativa suele estar en valores confortables: entre $40\ \%\ y\ 60\ \%$ y en situaciones extremas: entre $30\ \%\ y\ 70\ \%$.
    * Un tercer atributo con valores aleatorios que simule otra variable del entorno, como por ejemplo: **nivel de ocupación** (0–50 personas), **concentración de $CO₂$** o un **índice ficticio de actividad**.

In [3]:
import numpy as np
import pandas as pd

def generarDatasetSala(n_instancias=1000):
    """
    Crea un dataset sintético para una sala inteligente.
    """
    # 1. Crear el índice de tiempo o instancias
    instancias = np.arange(n_instancias)

    # 2. Generar Temperatura ambiente (12 - 30 °C)
    # Cubre tanto el rango normal (20-24) como situaciones no óptimas.
    temperatura = np.random.uniform(12, 30, size=n_instancias)

    # 3. Generar Humedad relativa (30 % - 70 %)
    # Cubre el rango confortable (40-60) y situaciones extremas.
    humedad = np.random.uniform(30, 70, size=n_instancias)

    # 4. Generar Nivel de Ocupación (0 - 50 personas)
    ocupacion = np.random.uniform(0, 50, size=n_instancias).astype(int)

    # Crear un DataFrame para facilitar el análisis posterior
    df = pd.DataFrame({
        'Instante': instancias,
        'Temperatura_C': temperatura,
        'Humedad_p': humedad,
        'Ocupacion_personas': ocupacion
    })

    return df

# Generar y mostrar los primeros resultados
dataset_sala = generarDatasetSala(1000)
print("--- Dataset Generado (Primeras 5 filas) ---")
print(dataset_sala.head())

--- Dataset Generado (Primeras 5 filas) ---
   Instante  Temperatura_C  Humedad_p  Ocupacion_personas
0         0      15.553272  40.450853                   4
1         1      27.893764  52.665819                  27
2         2      16.271227  46.730543                  17
3         3      13.291047  35.701941                  13
4         4      29.460719  35.497369                  31


---

2. Para cada instancia, calcula la probabilidad de disconfort térmico, teniendo en cuenta los valores de temperatura y humedad. A mayor desviación respecto a los rangos de confort, mayor debe ser la probabilidad asignada.

In [ ]:
def calcular_probabilidad_disconfort(df):
    """
    Calcula la probabilidad de disconfort térmico basada en la
    desviación de los rangos ideales (Temp: 22, Hum: 50).
    """
    # 1. Calcular desviaciones absolutas respecto al punto medio ideal
    desv_temp = np.abs(df['Temperatura_C'] - 22)
    desv_hum = np.abs(df['Humedad_p'] - 50)

    # 2. Definir una combinación lineal (z) para la sigmoide
    # b1 y b2 determinan qué tan rápido sube la probabilidad al alejarse del centro
    # a es el sesgo (bias) que ajusta el punto de inicio
    b1 = 0.8  # Peso para la temperatura
    b2 = 0.1  # Peso para la humedad
    a = -4.5  # Sesgo para que en el centro la probabilidad sea baja

    z = b1 * desv_temp + b2 * desv_hum + a

    # 3. Aplicar la Función Sigmoide (Regresión Logística)
    # P(Y=1) = 1 / (1 + e^-z)
    probabilidad = 1 / (1 + np.exp(-z))

    return probabilidad

# Aplicar al dataset anterior
dataset_sala['Probabilidad_Disconfort'] = calcular_probabilidad_disconfort(dataset_sala)

# Mostrar ejemplos de extremos y centro
print("--- Muestra de Probabilidades Calculadas ---")
print(dataset_sala[['Temperatura_C', 'Humedad_p', 'Probabilidad_Disconfort']].head(10))

In [6]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# 1. Preparar las variables independientes (Features)
# Calculamos la desviacin absoluta como entrada para el modelo
X = np.column_stack([
    np.abs(dataset_sala['Temperatura_C'] - 22),
    np.abs(dataset_sala['Humedad_p'] - 50)
])

# 2. Generar una variable objetivo (Target) para el entrenamiento
# Usamos la probabilidad previa como base para crear clases 0 (Confort) o 1 (Disconfort)
y = (dataset_sala['Probabilidad_Disconfort'] > 0.5).astype(int)

# 3. Inicializar y ajustar el modelo de Regresión Logística
# C=1.0 es la regularización por defecto
clf = LogisticRegression(solver='liblinear')
clf.fit(X, y)

# 4. Predecir probabilidades utilizando el modelo de Scikit-Learn
# predict_proba devuelve [prob_clase_0, prob_clase_1]
probabilidades_sklearn = clf.predict_proba(X)[:, 1]

dataset_sala['Prob_LogReg_Sklearn'] = probabilidades_sklearn

print("--- Coeficientes del Modelo (Nivel Anal3tico) ---")
print(f"Intersecci3n (Bias): {clf.intercept_[0]:.4f}")
print(f"Coeficientes (Temp, Hum): {clf.coef_[0]}")
display(dataset_sala[['Temperatura_C', 'Humedad_p', 'Prob_LogReg_Sklearn']].head(10))

--- Coeficientes del Modelo (Nivel Anal3tico) ---
Intersecci3n (Bias): -9.9487
Coeficientes (Temp, Hum): [1.82368113 0.21531157]


,Temperatura_C,Humedad_p,Prob_LogReg_Sklearn
0,21.071716,69.508393,0.017034
1,27.600843,30.302106,0.989085
2,25.780891,60.616643,0.316993
3,19.596032,33.567355,0.116455
4,22.557324,47.100545,0.000246
5,16.050748,37.772024,0.971632
6,17.409988,55.199271,0.387349
7,20.473388,52.516930,0.001328
8,13.522213,34.001206,0.999871
9,14.836794,55.958298,0.987843


### Comparativa: Manual vs. Autom&aacute;tico (Sklearn)

| Caracter&iacute;stica | Manual (Heur&iacute;stico) | LogisticRegression (Anal&iacute;tico) |
| :--- | :--- | :--- |
| **Origen de Pesos** | Definidos por el programador (`b1=0.8`) | Calculados optimizando el error |
| **Sesgo (Bias)** | Fijo (`a=-4.5`) | Ajustado seg&uacute;n la distribuci&oacute;n real |
| **Flexibilidad** | R&iacute;gida | Se adapta a nuevos datos |

In [7]:
# Comparaci&oacute;n de pesos
print("--- PESOS MANUALES ---")
print(f"Bias: -4.5")
print(f"Coeficiente Temp: 0.8")
print(f"Coeficiente Hum: 0.1")

print("\n--- PESOS APRENDIDOS POR SKLEARN ---")
print(f"Bias: {clf.intercept_[0]:.4f}")
print(f"Coeficiente Temp: {clf.coef_[0][0]:.4f}")
print(f"Coeficiente Hum: {clf.coef_[0][1]:.4f}")

# Visualizar si las probabilidades son similares
display(dataset_sala[['Probabilidad_Disconfort', 'Prob_LogReg_Sklearn']].head(10))

--- PESOS MANUALES ---
Bias: -4.5
Coeficiente Temp: 0.8
Coeficiente Hum: 0.1

--- PESOS APRENDIDOS POR SKLEARN ---
Bias: -9.9487
Coeficiente Temp: 1.8237
Coeficiente Hum: 0.2153


,Probabilidad_Disconfort,Prob_LogReg_Sklearn
0,0.141058,0.017034
1,0.875497,0.989085
2,0.398044,0.316993
3,0.282203,0.116455
4,0.022661,0.000246
5,0.814905,0.971632
6,0.423588,0.387349
7,0.046220,0.001328
8,0.979809,0.999871
9,0.861332,0.987843


---

3. En base a un **umbral**, determina para cada instancia la clase:
    * 1 → existe riesgo de disconfort térmico
    * 0 → no existe riesgo de disconfort térmico

In [ ]:
def asignar_clase_disconfort(df, umbral=0.5):
    """
    Asigna la clase binaria basándose en un umbral de probabilidad.
    """
    # Aplicamos la lógica: 1 si prob > umbral, de lo contrario 0
    # Usamos astype(int) para convertir los booleanos True/False a 1/0
    return (df['Probabilidad_Disconfort'] > umbral).astype(int)

# Aplicar al dataset
dataset_sala['Clase_Disconfort'] = asignar_clase_disconfort(dataset_sala)

# Mostrar el balance de las clases generadas
conteo_clases = dataset_sala['Clase_Disconfort'].value_counts()
print("--- Distribución de Clases ---")
print(f"Sin Riesgo (0): {conteo_clases[0]} instancias")
print(f"Con Riesgo (1): {conteo_clases[1]} instancias")

# Visualizar una muestra representativa
print("\n--- Muestra del Dataset Final ---")
print(dataset_sala[['Temperatura_C', 'Humedad_p', 'Probabilidad_Disconfort', 'Clase_Disconfort']].head(10))

--- Distribución de Clases ---
Sin Riesgo (0): 496 instancias
Con Riesgo (1): 504 instancias

--- Muestra del Dataset Final ---
   Temperatura_C  Humedad_p  Probabilidad_Disconfort  Clase_Disconfort
0      19.411450  50.040757                 0.081280                 0
1      28.009682  38.428779                 0.812276                 1
2      13.159281  52.482200                 0.943789                 1
3      15.556443  63.081110                 0.876853                 1
4      21.650841  42.195152                 0.031063                 0
5      14.071276  37.804968                 0.955318                 1
6      15.037271  45.182988                 0.825186                 1
7      17.701897  50.855529                 0.273693                 0
8      19.885640  44.140042                 0.097747                 0
9      17.162061  68.360905                 0.769669                 1
